In [ ]:

from ifc.config import TRAIN_FILE, TEST_FILE, TARGET, CLASSES, ID_COLS, CATEGORICAL_COLS, NUMERICAL_COLS, DROP_COLS, SEED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-darkgrid")
np.random.seed(SEED)

train_df = pd.read_csv(TRAIN_FILE)
test_df  = pd.read_csv(TEST_FILE)
train_df.drop(DROP_COLS,axis=1, inplace=True)

print(train_df.shape)
print(test_df.shape)

### 1. Dataset Structure
- Column types and non-null counts
- Temporal split sanity check (train 2018–2021, test 2022–2023)
- Unique companies and rows per company

In [ ]:
train_df.head()

In [ ]:

train_df.info()


In [ ]:

print("Train years:", sorted(train_df["fiscal_year"].unique()))
print("Test years: ", sorted(test_df["fiscal_year"].unique()))


In [ ]:

print("Unique companies — train:", train_df["company_id"].nunique())
print("Unique companies — test: ", test_df["company_id"].nunique())

rows_per_company = train_df.groupby("company_id").size().value_counts().sort_index()
print("\nRows per company:\n", rows_per_company)


In [ ]:

data_types = train_df.dtypes.value_counts().reset_index()
data_types.columns = ['Dtype', 'Count']

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

ax = sns.barplot(
    data=data_types, 
    x='Count', 
    y='Dtype', 
    palette='magma',
    hue='Dtype',
    legend=False
)

ax.bar_label(ax.containers[0], padding=3, fontsize=12)

plt.title('Distribution of Column Data Types', fontsize=16, pad=20, fontweight='bold')
plt.xlabel('Number of Columns', fontsize=12)
plt.ylabel('Data Type', fontsize=12)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

In [ ]:

full_years = set([2018, 2019, 2020, 2021])

company_years = train_df.groupby("company_id")["fiscal_year"].apply(set)
incomplete = company_years[company_years.apply(len) < 4]

print(f"Companies with incomplete history: {len(incomplete)} / {train_df['company_id'].nunique()}")

summary = pd.DataFrame({
    "years_present": incomplete,
    "years_missing": incomplete.apply(lambda y: sorted(full_years - y)),
    "n_years": incomplete.apply(len)
})

print("\nMissing year patterns:")
print(summary["years_missing"].value_counts())


In [ ]:
# Visualise: which fiscal year is most often absent
from itertools import chain

all_missing = list(chain.from_iterable(summary["years_missing"]))
missing_counts = pd.Series(all_missing).value_counts().sort_index()

missing_counts.plot(kind="bar", color="#e74c3c", edgecolor="white", figsize=(7, 4))
plt.title("Frequency of Missing Fiscal Year (incomplete companies)")
plt.xlabel("Fiscal Year"); plt.ylabel("Count")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


In [ ]:
# Are incomplete companies concentrated in a specific class or sector?
incomplete_ids = incomplete.index
incomplete_df = train_df[train_df["company_id"].isin(incomplete_ids)]

print("Class distribution — incomplete companies:")
print(incomplete_df.drop_duplicates("company_id")[TARGET].value_counts(normalize=True).round(3))

print("\nTop sectors — incomplete companies:")
print(incomplete_df.drop_duplicates("company_id")["ateco_sector"].value_counts().head(10))


In [ ]:
# Check for year gaps within a company's history (e.g. 2018, 2020 — missing 2019)
def find_gaps(years):
    years = sorted(years)
    gaps = []
    for i in range(len(years) - 1):
        expected = years[i] + 1
        actual = years[i + 1]
        if actual != expected:
            gaps.append((years[i], actual))  # (last seen, next seen)
    return gaps

company_gaps = company_years.apply(find_gaps)
companies_with_gaps = company_gaps[company_gaps.apply(len) > 0]

print(f"Companies with year gaps: {len(companies_with_gaps)} / {train_df['company_id'].nunique()}")
print("\nGap patterns (from_year → to_year):")
print(companies_with_gaps.explode().value_counts())


In [ ]:
# For each incomplete company, check if missing years are at the start or end
def missing_position(years):
    years = sorted(years)
    missing = sorted(full_years - set(years))
    positions = []
    for y in missing:
        if y < years[0]:
            positions.append("start")   # missing early years → entered dataset late
        elif y > years[-1]:
            positions.append("end")     # missing late years → exited dataset early
        else:
            positions.append("middle")  # impossible given no gaps, but safety check
    return positions

summary["missing_position"] = summary["years_present"].apply(missing_position)
print(summary["missing_position"].explode().value_counts())


In [ ]:
# Do companies missing 2021 skew toward D?
missing_2021 = summary[summary["years_missing"].apply(lambda x: 2021 in x)].index
print("Class dist — missing 2021 (early exits):")
print(train_df[train_df["company_id"].isin(missing_2021)][TARGET].value_counts(normalize=True).round(3))

# Compare to full dataset
print("\nClass dist — full dataset:")
print(train_df[TARGET].value_counts(normalize=True).round(3))



- Scalers and imputers must be **fit on train only**, then applied to test
- Lag features must use **only past data** (year `t-1` → year `t`)
- Never use features from year `t+1` to predict class at year `t`

---

### 🔍 Finding: Incomplete Companies are Not Missing at Random

**104 companies have fewer than 4 years** in the training set:

| Pattern | Count | Interpretation |
|---|---|---|
| Missing at **end** (early exits) | 130 observations | Company stopped filing — likely distress |
| Missing at **start** (late entrants) | 38 observations | Young company, entered after 2018 |

**Early exits are strongly associated with class D:**

| Class | Early Exits | Full Dataset |
|-------|-------------|--------------|
| A     | 2.9%        | 8.5%         |
| B     | 13.1%       | 59.3%        |
| C     | 19.0%       | 23.2%        |
| D     | **65.0%**   | **8.9%**     |

65% of companies that disappeared before 2021 were class D — vs only 8.9% in the full dataset.  
This is **informative missingness**, not random. These companies stopped filing due to severe financial distress.

> **Survivorship bias**: companies present in the test set (2022–2023) are survivors by definition.  
> The model may underestimate D risk for companies that will exit mid-period. This is a known limitation.

---

### Time Dimension as an Asset: Lag Features

Since most companies have 4 consecutive years of contiguous data, we can engineer **trend features**  
that capture whether a company's financial health is improving or deteriorating.

A company with ROE dropping from `0.4 → 0.1` is riskier than one stable at `0.15`,  
even though the current value is higher. Raw snapshots miss this signal.

**Features to engineer (computed at year `t` using year `t-1`)**:

| Feature | Formula | Signal |
|---|---|---|
| `roe_prev` | `roe` at `t-1` | Profitability baseline |
| `roi_prev` | `roi` at `t-1` | Efficiency baseline |
| `current_ratio_prev` | `current_ratio` at `t-1` | Liquidity baseline |
| `roe_yoy` | `roe_t - roe_{t-1}` | Profitability trend |
| `roi_yoy` | `roi_t - roi_{t-1}` | Efficiency trend |
| `leverage_trend` | `leverage_t - leverage_{t-1}` | Increasing debt signal |
| `equity_growth` | `(equity_t - equity_{t-1}) / abs(equity_{t-1})` | Capital erosion signal |
| `is_last_observation` | 1 if final row for company | Exit/distress signal |
| `n_years_in_panel` | count of years in dataset | Short history = higher risk |

> Late entrants (38 companies missing 2018) will have `NaN` lag features on their first row → imputed with median during preprocessing.  
> `is_last_observation` and `n_years_in_panel` must be computed **before** the train/test split to avoid leakage.

---

### Cross-Validation Strategy

Standard k-fold CV is **not valid** here because it would leak future data into training.  
Use **time-based expanding window CV** on the training set:

| Fold | Train | Validation |
|---|---|---|
| 1 | 2018 | 2019 |
| 2 | 2018–2019 | 2020 |
| 3 | 2018–2020 | 2021 ← most important |

Use `sklearn.model_selection.TimeSeriesSplit` or build manually on `fiscal_year`.  
**Never shuffle** when splitting.

---

### Full Modeling Flow
Sort by company_id + fiscal_year

Compute panel features: is_last_observation, n_years_in_panel

Engineer lag features (shift within each company group)

Temporal train/val split for CV (no shuffle)

Fit scaler + imputer on train only → transform val and test

Train classifier on all train years (2018–2021)

Predict on test set (2022–2023)

Evaluate: Weighted F1 (primary), Confusion Matrix, Per-class Precision/Recall

## 2. Target Variable Analysis
- Class counts and percentages (A/B/C/D)
- Class distribution per `fiscal_year` — detect COVID drift in 2020–2021
- Class distribution by `ateco_sector`, `legal_form`, `region`

In [ ]:

counts = train_df[TARGET].value_counts().reindex(CLASSES)
pcts   = train_df[TARGET].value_counts(normalize=True).reindex(CLASSES) * 100

print(pd.concat([counts, pcts.round(2)], axis=1, keys=["count", "%"]))


In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
palette = {"A": "#2ecc71", "B": "#3498db", "C": "#e67e22", "D": "#e74c3c"}
bars = ax.bar(CLASSES, counts, color=[palette[c] for c in CLASSES], edgecolor="white")
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{pct:.1f}%", ha="center", fontsize=11)
ax.set_title("Target Class Distribution — financial_health_class")
ax.set_xlabel("Class"); ax.set_ylabel("Count")
plt.tight_layout(); plt.show()


In [ ]:
# 2.3 Class distribution per fiscal year — detect COVID drift
yearly = pd.crosstab(train_df["fiscal_year"], train_df[TARGET], normalize="index")[CLASSES]

yearly.plot(kind="bar", stacked=True, figsize=(8, 4),
            color=[palette[c] for c in CLASSES], edgecolor="white")
plt.title("Class Distribution by Fiscal Year")
plt.ylabel("Proportion"); plt.xlabel("Fiscal Year")
plt.xticks(rotation=0); plt.legend(title="Class", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

# Raw numbers per year
print(pd.crosstab(train_df["fiscal_year"], train_df[TARGET]))


In [ ]:

legal = pd.crosstab(train_df["legal_form"], train_df[TARGET], normalize="index")[CLASSES]
legal.plot(kind="bar", stacked=True, figsize=(9, 4),
           color=[palette[c] for c in CLASSES], edgecolor="white")
plt.title("Class Distribution by Legal Form")
plt.ylabel("Proportion"); plt.xticks(rotation=0)
plt.legend(title="Class", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

legal

In [ ]:
# 2.5 Class distribution by region (top 10 regions by count)
top_regions = train_df["region"].value_counts().head(10).index
region_df = train_df[train_df["region"].isin(top_regions)]

region = pd.crosstab(region_df["region"], region_df[TARGET], normalize="index")[CLASSES]
region.plot(kind="bar", stacked=True, figsize=(12, 4),
            color=[palette[c] for c in CLASSES], edgecolor="white")
plt.title("Class Distribution by Region (Top 10)")
plt.ylabel("Proportion"); plt.xticks(rotation=45)
plt.legend(title="Class", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

region

In [ ]:
# 2.6 Class distribution by ateco_sector (top 10 sectors by count)
top_sectors = train_df["ateco_sector"].value_counts().head(10).index
sector_df = train_df[train_df["ateco_sector"].isin(top_sectors)]

sector = pd.crosstab(sector_df["ateco_sector"], sector_df[TARGET], normalize="index")[CLASSES]
sector.plot(kind="bar", stacked=True, figsize=(12, 4),
            color=[palette[c] for c in CLASSES], edgecolor="white")
plt.title("Class Distribution by ATECO Sector (Top 10)")
plt.ylabel("Proportion"); plt.xticks(rotation=0)
plt.legend(title="Class", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

sector

## Section 2 — Target Variable Analysis: Findings

### 2.1 Class Imbalance

The dataset is **moderately imbalanced**:

| Class | Count | % |
|-------|-------|---|
| A — Excellent | 1,003 | 8.5% |
| B — Good | 7,017 | 59.3% |
| C — Moderate risk | 2,750 | 23.3% |
| D — High risk | 1,058 | 8.9% |

B dominates the dataset. A and D are the minority classes and the hardest to classify correctly.
This means **accuracy is a misleading metric** — a model predicting B for everything would reach 59% accuracy.
→ Use **Weighted F1** as primary metric. Apply `class_weight="balanced"` in all models.

---

### 2.2 No COVID Drift Detected

Class distribution is **remarkably stable** across all 4 years:

| Year | A | B | C | D |
|------|---|---|---|---|
| 2018 | 254 | 1815 | 640 | 252 |
| 2019 | 260 | 1766 | 691 | 262 |
| 2020 | 239 | 1745 | 695 | 277 |
| 2021 | 250 | 1691 | 724 | 267 |

No significant spike in D during 2020–2021 despite COVID-19.
C shows a mild upward trend (+13% from 2018 to 2021) but no structural shift.
→ `fiscal_year` is **not a strong standalone predictor** — but keep it as a feature
to capture any subtle macro trend the model can learn.

---

### 2.3 Legal Form has Minimal Predictive Power

All legal forms (SRL, SPA, SAS, SNC, SAPA) show nearly identical class distributions,
all within ±2% of the global baseline.
→ `legal_form` is a **weak feature**. Encode it but don't expect high importance.

---

### 2.4 Region has Minimal Predictive Power

Regional differences are small. Notable exceptions:
- **Liguria** has the highest C rate (27.7%) and lowest B rate (53.9%) — slightly riskier
- **Piemonte** has the highest B rate (62.3%) — slightly healthier
- **Veneto** has the lowest A rate (7.0%)

Differences are marginal (~5% range across regions).
→ `region` is a **weak feature**. Consider grouping into macro-areas (Nord/Centro/Sud)
to reduce cardinality without losing the small signal.

---

### 2.5 ATECO Sector has Moderate Predictive Power

More meaningful variation than region or legal form:
- **Construction (41, 43)** — highest C rates (30%, 28.8%) and lowest A rates (~6%) →
  capital-intensive sectors with thin margins and high debt
- **Wholesale trade (46, 47)** — highest A rates (10.2%, 11.4%) → healthier cash flow
  businesses
- **IT services (62)** — lowest D rate (6.4%) → knowledge-based sectors are more resilient
- **Food manufacturing (10)** — highest D rate (10.5%)

→ `ateco_sector` is a **meaningful feature**. Keep at full granularity.
Also engineer **sector-relative ratios** (e.g., `roe - sector_median_roe`) to
capture how a company performs vs its peers.

---

### Preprocessing Decisions from Section 2

1. **Never use accuracy** — always Weighted F1
2. **Apply `class_weight="balanced"`** in all classifiers
3. **Keep `fiscal_year`** as a feature (mild macro signal)
4. **Encode `legal_form` and `region`** but expect low importance
5. **Keep `ateco_sector` at full granularity** — engineer sector-relative ratio features
6. **Consider macro-region grouping** (Nord/Centro/Sud) as an additional feature


### 3. Missing Values
- Missing count and % per column
- Structural vs random: `roe`/`leverage` missing when equity = 0
- `province` missingness — correlated with class?
- Missingness heatmap by `fiscal_year`

In [ ]:
# 3.1 Missing count and % per column
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pct = (missing / len(train_df) * 100).round(2)
print(pd.concat([missing, pct], axis=1, keys=["count", "%"]))


In [ ]:
# 3.2 Is roe/leverage missingness structural (equity = 0)?
mask_roe = train_df["roe"].isnull()
print(f"ROE null rows: {mask_roe.sum()}")
print(f"Of which equity = 0: {(train_df.loc[mask_roe, 'shareholders_equity'] == 0).sum()}")
print(f"Of which equity < 0: {(train_df.loc[mask_roe, 'shareholders_equity'] < 0).sum()}")
print(f"\nLeverage null rows: {train_df['leverage'].isnull().sum()}")
print(f"Same rows as ROE null: {(train_df['roe'].isnull() == train_df['leverage'].isnull()).all()}")


In [ ]:
# 3.3 Class distribution where roe/leverage is null
print("Class dist — where ROE is null:")
print(train_df[mask_roe][TARGET].value_counts(normalize=True).round(3))

print("\nClass dist — full dataset:")
print(train_df[TARGET].value_counts(normalize=True).round(3))


In [ ]:
# How prevalent is negative equity across classes?
train_df["negative_equity"] = (train_df["shareholders_equity"] < 0).astype(int)

print("Negative equity rate by class:")
print(train_df.groupby(TARGET)["negative_equity"].mean().round(3) * 100)

print("\nNegative equity count by class:")
print(train_df.groupby(TARGET)["negative_equity"].sum())


In [ ]:
# Of all class D rows, what % have negative equity?
d_rows = train_df[train_df[TARGET] == "D"]
print(f"Class D rows with negative equity: {d_rows['negative_equity'].sum()} / {len(d_rows)}")
print(f"That is {d_rows['negative_equity'].mean()*100:.1f}% of all D rows")

train_df.drop(columns="negative_equity", inplace=True)


In [ ]:
# 3.4 Province missingness — correlated with class?
train_df["province_missing"] = train_df["province"].isnull().astype(int)

print("Class dist — province missing:")
print(train_df[train_df["province_missing"] == 1][TARGET].value_counts(normalize=True).round(3))

print("\nProvince missing rate % by class:")
print(f"{train_df.groupby(TARGET)["province_missing"].mean().round(3)*100}")

train_df.drop(columns="province_missing", inplace=True)


In [ ]:
# 3.5 Missingness by fiscal year
missing_by_year = train_df.groupby("fiscal_year")[["roe", "leverage", "province"]].apply(
    lambda x: x.isnull().sum()
)
print(missing_by_year)



## Section 3 — Missing Values: Findings

### 3.1 Overview

Only 3 columns have missing values:

| Column | Missing | % | Type |
|--------|---------|---|------|
| `province` | 919 | 7.77% | Random (MCAR) |
| `roe` | 45 | 0.38% | Structural |
| `leverage` | 45 | 0.38% | Structural |

The dataset is **exceptionally clean** — no balance sheet or income statement
items are missing.

---

### 3.2 ROE & Leverage — Structural Missingness (100% signal for class D)

All 45 null `roe` and `leverage` rows belong to the **exact same observations**
and all have **negative shareholders equity**.

When equity is negative:
- `roe = net_profit / equity` → mathematically valid but financially meaningless
- `leverage = total_debt / equity` → negative leverage is uninterpretable

The dataset correctly marks these as `NaN` rather than showing a misleading value.

**Critical finding: 100% of rows where ROE is null are class D.**

This means null `roe`/`leverage` is not a data quality issue —
it is the **strongest possible signal of distress** in the entire dataset.

→ **Do NOT impute these with median.** Instead:
1. Create `roe_is_null` binary feature (= 1 when equity is negative)
2. Impute the raw value with an **extreme negative sentinel** (e.g. `-99`)
   to preserve the distress signal for tree-based models
3. For linear models, impute with a value well below the 1st percentile

---

### 3.3 Negative Equity Deep Dive

Negative equity is **exclusive to class D** — zero cases in A, B, or C.
However it only covers **4.3% of all D rows** (45 / 1,058).

| Finding | Implication |
|---|---|
| Negative equity never appears in A/B/C | `roe_is_null` is a **perfect precision** signal — when it fires, it is always D |
| Only 4.3% of D rows have negative equity | `roe_is_null` has **very low recall** — catches only a small fraction of distressed companies |

This means negative equity is a **late-stage distress indicator** — the company
is already deep in trouble. The model must detect D much earlier using
deterioration signals from other features (leverage trend, ROI decline,
liquidity squeeze) before equity turns negative.

→ `roe_is_null` is a **high-precision, low-recall** feature. Keep it — it will
likely become a top split in tree-based models for class D. The bulk of D
detection must come from financial ratio trends.

---

### 3.4 Province — Missing Completely at Random (MCAR)

Province missingness rate is **uniform across all classes** (~7.2–7.9%):

| Class | Province missing % |
|-------|--------------------|
| A | 7.2% |
| B | 7.9% |
| C | 7.5% |
| D | 7.9% |

No correlation with class, sector, or fiscal year (stable ~229 missing per year).
This is **Missing Completely at Random (MCAR)** — likely companies that registered
without a province code in the source database.

→ **Impute with a new category `"UNKNOWN"`** — do not drop rows or use
mode imputation, as province is a categorical with 107 levels and the
missing pattern carries no financial signal.

---

### 3.5 Missingness is Stable Across Years

No year concentration — ~11–12 null `roe`/`leverage` and ~229 null `province`
per year consistently. No COVID-related reporting gaps detected.

---

### Preprocessing Decisions from Section 3

| Column | Treatment | Rationale |
|--------|-----------|-----------|
| `roe`, `leverage` | Add `roe_is_null` binary flag → impute with sentinel `-99` | Perfect D signal, must preserve |
| `province` | Impute with `"UNKNOWN"` category | MCAR, no financial signal |
| All other columns | No imputation needed | Dataset is complete |

### 4. Descriptive Statistics
- Summary stats for all numerical columns
- Log-scale histograms for balance sheet items (heavy right skew expected)
- Check impossible values: negative revenue, `debt_to_assets > 1`


In [ ]:
# 4.1 Summary statistics for all numerical columns
train_df[NUMERICAL_COLS].describe().T.round(2)


In [ ]:
# 4.2 Log-scale histograms for balance sheet and income statement items
bs_cols = ["total_assets", "production_value", "total_debt",
           "shareholders_equity", "current_assets", "total_fixed_assets"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), bs_cols):
    data = train_df[col].clip(lower=1)  # clip negatives before log
    ax.hist(np.log1p(data), bins=40, color="#3498db", edgecolor="white")
    ax.set_title(f"log({col})")
    ax.set_xlabel("log value"); ax.set_ylabel("count")
plt.suptitle("Balance Sheet & Income Statement — Log Distributions", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# 4.3 Log-scale histograms for financial ratios
ratio_cols = ["roe", "roi", "current_ratio", "quick_ratio",
              "debt_to_assets", "profit_margin", "leverage"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), ratio_cols):
    data = train_df[col].dropna()
    ax.hist(data, bins=50, color="#e67e22", edgecolor="white")
    ax.set_title(col)
    ax.axvline(data.median(), color="red", linestyle="--", label="median")
    ax.legend(fontsize=8)
plt.suptitle("Financial Ratios — Distributions", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# 4.4 Check impossible values

# Negative production value (revenue cannot be negative)
neg_revenue = (train_df["production_value"] < 0).sum()
print(f"Negative production_value: {neg_revenue}")

# Negative production costs
neg_costs = (train_df["production_costs"] < 0).sum()
print(f"Negative production_costs: {neg_costs}")

# debt_to_assets > 1 (insolvent — liabilities exceed assets)
dta_over1 = (train_df["debt_to_assets"] > 1).sum()
print(f"debt_to_assets > 1 (insolvent): {dta_over1} ({dta_over1/len(train_df)*100:.2f}%)")

# current_ratio < 0.5 (severe liquidity stress)
cr_low = (train_df["current_ratio"] < 0.5).sum()
print(f"current_ratio < 0.5 (severe stress): {cr_low}")

# leverage > 50 (likely error or extreme distress)
lev_extreme = (train_df["leverage"] > 50).sum()
print(f"leverage > 50 (extreme): {lev_extreme}")

# Accounting identity violation: total_assets ≠ equity + debt
train_df["identity_gap"] = (
    train_df["total_assets"] - train_df["shareholders_equity"] - train_df["total_debt"]
).abs()
print(f"\nMax accounting identity gap: {train_df['identity_gap'].max():,.0f}")
print(f"Rows with gap > 1000: {(train_df['identity_gap'] > 1000).sum()}")
train_df.drop(columns="identity_gap", inplace=True)


In [ ]:
# 4.5 Class distribution for impossible/extreme value rows
print("Class dist — debt_to_assets > 1:")
print(train_df[train_df["debt_to_assets"] > 1][TARGET].value_counts(normalize=True).round(3))

print("\nClass dist — leverage > 50:")
print(train_df[train_df["leverage"] > 50][TARGET].value_counts(normalize=True).round(3))

print("\nClass dist — current_ratio < 0.5:")
print(train_df[train_df["current_ratio"] < 0.5][TARGET].value_counts(normalize=True).round(3))


In [ ]:
# Check: what does a "typical" company look like?
sample = train_df[train_df["company_id"] == "COMP_00001"].iloc[0]
print(f"Production value: €{sample['production_value']:,.0f}")
print(f"Total assets:     €{sample['total_assets']:,.0f}")
print(f"Net profit:       €{sample['net_profit_loss']:,.0f}")
print(f"Employees proxy — production_value / 50000: {sample['production_value']/50000:.0f}")


In [ ]:
# Distribution of company sizes — how many "micro" companies?
size_buckets = pd.cut(train_df["production_value"],
    bins=[0, 2e6, 10e6, 50e6, 200e6, 1e9, float("inf")],
    labels=["<2M", "2-10M", "10-50M", "50-200M", "200M-1B", ">1B"])
print(size_buckets.value_counts().sort_index())


In [ ]:
# Test 1: Cross-check accounting identity precision
# If data is scaled by integer factor, the identity gap should still be 0
# (which we confirmed — max gap = 0) ✅ consistent with uniform scaling

# Test 2: Check if ratio formulas hold with the raw values
# roe = net_profit_loss / shareholders_equity
train_df["roe_computed"] = train_df["net_profit_loss"] / train_df["shareholders_equity"]
diff = (train_df["roe_computed"] - train_df["roe"]).abs()
print("Max ROE recomputation error:", diff.max().round(6))
print("Mean ROE recomputation error:", diff.mean().round(6))
train_df.drop(columns="roe_computed", inplace=True)


In [ ]:
# Test 3: Check if debt_to_assets holds
train_df["dta_computed"] = train_df["total_debt"] / train_df["total_assets"]
diff2 = (train_df["dta_computed"] - train_df["debt_to_assets"]).abs()
print("Max DTA recomputation error:", diff2.max().round(6))
train_df.drop(columns="dta_computed", inplace=True)


In [ ]:
# Test 4: Compare production_value / production_costs ratio
# For Italian companies, operating cost ratio is typically 85-98%
# If scaled, this ratio is unaffected
op_cost_ratio = (train_df["production_costs"] / train_df["production_value"])
print("Operating cost ratio stats:")
print(op_cost_ratio.describe().round(3))


In [ ]:
# Test 5: Check profit_margin formula
train_df["pm_computed"] = train_df["net_profit_loss"] / train_df["production_value"]
diff3 = (train_df["pm_computed"] - train_df["profit_margin"]).abs()
print("Max profit_margin recomputation error:", diff3.max().round(6))
train_df.drop(columns="pm_computed", inplace=True)


In [ ]:
# Estimate scale factor by comparing to typical Italian SME sizes
# Typical Italian SRL: revenue €1M - €50M, assets €500K - €20M
# Our median: production_value = €804M, total_assets = €521M

typical_sme_revenue = 5_000_000      # €5M — typical Italian SRL
our_median_revenue = 804_001_932     # from describe()

scale_factor = our_median_revenue / typical_sme_revenue
print(f"Estimated scale factor: {scale_factor:,.0f}x")

# Cross-check with assets
typical_sme_assets = 2_000_000       # €2M
our_median_assets = 521_237_912
print(f"Asset-based scale factor: {our_median_assets / typical_sme_assets:,.0f}x")



→ These are legitimate extreme distress observations — do not drop them.
→ Winsorize `leverage` at 99th percentile but preserve the `roe_is_null` flag.

---

### 4.5 Critical Finding: Dataset is Synthetically Generated

#### Evidence

Ratio recomputation tests confirm perfect internal consistency:

| Test | Max error | Verdict |
|---|---|---|
| ROE recomputation | 5e-05 (float rounding) | ✅ Perfect |
| DTA recomputation | 5e-05 (float rounding) | ✅ Perfect |
| Profit margin recomputation | 5e-05 (float rounding) | ✅ Perfect |
| Accounting identity gap | Exactly 0 | ✅ Perfect |

**A perfect accounting identity gap of 0 across 11,828 rows is impossible
in real financial filing data.** Real statements always contain rounding
differences between filed totals. This is the clearest proof of synthetic
generation.

#### Scale Investigation

Attempting to recover a uniform scale factor:

| Reference | Implied scale vs typical Italian SRL |
|---|---|
| Revenue-based | ~161x |
| Asset-based | ~261x |

The **inconsistency rules out uniform scaling**. The generator did not
simply multiply real values by a constant. Instead it sampled absolute
values from a large-cap size distribution while calibrating financial
ratios to realistic SME dynamics.

The operating cost ratio (median 92.4%, range 58.5%–113.4%) confirms
the ratio dynamics are realistic and match real Italian sector economics.

#### Implications for Modeling

| Feature type | Reliability | Modeling decision |
|---|---|---|
| Financial ratios (`roe`, `roi`, etc.) | ✅ Fully reliable | Primary feature set |
| Ratio trends (YoY changes) | ✅ Fully reliable | Engineer as lag features |
| Operating cost ratio | ✅ Realistic | Use as engineered feature |
| Absolute monetary values | ⚠️ Unrealistic size | Log-transform, use as relative proxy only |
| Size-based benchmarks | ❌ Avoid | Cannot compare to real companies |

→ **Ratios and their trends are the primary feature set.**
→ Absolute values kept only as **relative size proxies** via log transformation.
→ Synthetic nature acknowledged as a **limitation** in the final presentation.

---

### Preprocessing Decisions from Section 4

| Variable | Treatment | Rationale |
|---|---|---|
| Balance sheet & income statement | `np.log1p` transform | Correct right skew for linear/distance models |
| `shareholders_equity` | `np.log1p` on positive values only + flag negatives | Bimodal distribution |
| `roe` | Winsorize at 1st/99th percentile | Extreme left outliers (-39.2) |
| `leverage` | Winsorize at 99th percentile | 5 extreme outliers (max 101) |
| `roi`, `current_ratio`, `quick_ratio` | No transformation | Clean bounded distributions |
| `debt_to_assets`, `profit_margin` | No transformation | Already bounded ranges |
| Extreme distress rows (45 rows) | Keep | Legitimate class D signal |


### 5. Outlier Detection
- 1st/99th percentile bounds for all numerical columns
- Accounting identity check: `total_assets = equity + total_debt`
- Flag extreme `leverage` values (> 50)

In [ ]:
# 5.1 Winsorization bounds — 1st and 99th percentiles for all numerical columns
bounds = train_df[NUMERICAL_COLS].quantile([0.01, 0.99]).T
bounds.columns = ["p1", "p99"]
bounds["range"] = bounds["p99"] - bounds["p1"]
print(bounds.round(4))


In [ ]:
# 5.2 Count of outliers beyond bounds per column
outlier_counts = pd.DataFrame({
    "below_p1": (train_df[NUMERICAL_COLS] < bounds["p1"]).sum(),
    "above_p99": (train_df[NUMERICAL_COLS] > bounds["p99"]).sum(),
})
outlier_counts["total"] = outlier_counts["below_p1"] + outlier_counts["above_p99"]
outlier_counts["pct"] = (outlier_counts["total"] / len(train_df) * 100).round(2)
print(outlier_counts.sort_values("total", ascending=False))
